### imports & set up

In [12]:
import os
import sys
import json
import torch
import functools
import numpy as np
import pandas as pd
import gc
from typing import List
from torch import Tensor
from transformers import AutoTokenizer
from jaxtyping import Int
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import scipy.stats as stats
from scipy.stats import mannwhitneyu, norm
import dotenv
import os
from pathlib import Path
import shutil
import traceback
from huggingface_hub.constants import HF_HUB_CACHE
from transformer_lens.model_bridge import TransformerBridge as TransformerBridge
from pathlib import Path

In [2]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

In [3]:
print(torch.__version__) #should be 2.6 

2.6.0+cu124


In [4]:
# #load environment variables from .env file
# if not dotenv.load_dotenv():
#     print("Could not find dotenv file...")
# else:
#     print("Successfully loaded dotenv file. ", end="")
#     print(f"Environment PROJECT={os.environ.get('PROJECT')}")

#test

In [5]:
#set up paths
src_path = Path().resolve() / "../src"
sys.path.append(src_path)
project_root = src_path.resolve().parent  
sys.path.append(str(project_root))

In [11]:
#local imports from src 
import src.interp_toolkit as interp_toolkit

In [ ]:
# known issue with statsmodels means there is an issue with the following package 
#https://github.com/statsmodels/statsmodels/issues/9584
# import statsmodels.api as sm
# from statsmodels.formula.api import ols

In [ ]:
torch.set_grad_enabled(False)


In [ ]:
interp_toolkit.load_environment_variables()

Successfully loaded dotenv file. Environment PROJECT=my_project


In [ ]:
interp_toolkit.check_environment()

Hugging face cache inside the dockerfile lives at /home/aleshins/.cache
Hugging face token is hf_...Xso


### basic demo

##### load a model by name

In [13]:

#set its familty as a global variable -- this is only for purposes of prompt formatting
MODEL_FAMILY = "qwen" #options: "qwen", "llama", "mistral", "gemma"

#load a model 
model = interp_toolkit.load_model("Qwen/Qwen3.5-2B",device="cuda:1", model_family=MODEL_FAMILY)
    



Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Moving model to device:  cuda
Model 'Qwen/Qwen3.5-2B' loaded successfully on cuda:1.


##### format a prompt using a chat template

In [12]:
#TODO: write function to read in stimuli and prompts from json files later--this is a simple demo for now
#suppose this might be constructed from a list of stimuli read in from a file.


sample_prompt= 'What is the capital city of France?'
system_message = "You are a helpful assistant."

#format a prompt for this stimulus item
this_prompt = interp_toolkit.create_prompt(MODEL_FAMILY, system_message, sample_prompt)
print(this_prompt)

<|im_start|>system
You are a helpful assistant..<|im_end|>
<|im_start|>user
What is the capital city of France?<|im_end|>
<|im_start|>assistant



In [33]:
#what dimensions should I expect? 
print('model dimensions are:',model.cfg.d_model)
print('vocabulary size is: ',model.cfg.d_vocab)

model dimensions are: 2048
vocabulary size is:  248320


##### call a model to generate a response

In [13]:
interp_toolkit.generate(model, this_prompt, max_new_tokens=50)

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:02<00:00, 17.45it/s]


['<|im_start|>system\nYou are a helpful assistant..<|im_end|>\n<|im_start|>user\nWhat is the capital city of France?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nThe capital city of France is **Paris**. Located in the northern part of the country, it is situated on the banks of the Seine river and has been the political and cultural heart of the nation since the 12th century']

##### call a model and get activations and logits

In [ ]:
activations, logits= interp_toolkit.get_activations(model, this_prompt, layer=0, token_pos  = -1, hook_type="residual_pre")

#### mean difference vectors and projections

In [175]:
#made up example of stim inputs
positive_stimuli = ["The capital of France is Paris.", "The language in France is French.", "French cuisine involves wonderful pastries."]
negative_stimuli = ["The capital of Germany is Berlin.", "The people of Germany speak German.", "A famous German composer is Bach."]
token_indices_positive = [(-2,-1), (-2,-1), (-2,-1)]  # Example token indices for positive stimuli...just taking the last in each case
token_indices_negative = [(-2,-1), (-2,-1), (-2,-1)]  # Example token indices for negative stimuli
hook_type = 'residual_pre'
layer = None #if layer is None, you will get activations for all layers 
system_message = "You are a helpful assistant."

In [178]:
activation_vector = interp_toolkit.extract_activation_vector(model,
                                               model_family = interp_toolkit.MODEL_FAMILY, 
                                               positive_stimuli=positive_stimuli, 
                                               negative_stimuli=negative_stimuli,
                                               token_indices_positive=token_indices_positive, 
                                               token_indices_negative=token_indices_negative, 
                                               hook_type=hook_type, layer=layer,
                                               system_message=system_message)

In [180]:
activation_vector.ndim

2

In [181]:
activation_vector.shape #check that this makes sense! Dim 1 is layer, and the last dim should be the activations (d_model)

torch.Size([24, 2048])

In [186]:
#define stimuli to project onto the activation direction
stimuli = ["Bonjour!","Mais Ouis!", "Le Tour Eiffel","Shnitzel","Berlin Wall", "Einstein"] 

#project all stimuli onto each layer's activation direction, in corresponding layers only!
projections = interp_toolkit.project_stimuli_onto_direction(stimuli= stimuli, 
                                                     model=model,
                                                    activation_direction=activation_vector,
                                                    hook_type='residual_pre', 
                                                    token_ind=-1,
                                                    layer = None)

In [194]:
projections[10]

array([0.01354614, 0.01304708, 0.00467597, 0.00509103, 0.00254155,
       0.00508796], dtype=float32)

In [187]:
projections.shape #this should be: n_layers, n_stimuli.

(24, 6)

In [ ]:
#example of doing this for just a single layer, e.g. layer 10. Note that the activation direction is a 1D vector in this case, not a 2D array of all layers.
projections_1 = interp_toolkit.project_stimuli_onto_direction(stimuli= stimuli, 
                                                     model=model,
                                                    activation_direction=activation_vector[10,:],
                                                    hook_type='residual_pre', 
                                                    token_ind=-1,
                                                    layer = 10)

In [193]:
projections_1

array([0.01354614, 0.01304708, 0.00467597, 0.00509103, 0.00254155,
       0.00508796], dtype=float32)